In [9]:
import json
%matplotlib inline
import torch
import torch.nn as nn
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import re
from collections import Counter
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import string
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.metrics import mean_squared_error, confusion_matrix
from sklearn.model_selection import train_test_split
from helper import classToLabels, encode_sentence, pre_process_stem_sentence_array, correct_labels
from sklearn.metrics import classification_report
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.corpus import stopwords
import pandas as pd
from helper import nlp_pre_process_stem


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


In [10]:
class DocumentDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.y = Y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx][0].astype(np.int32)), self.y[idx], self.X[idx][1]


class LSTM_fixed_len(torch.nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, n_classes):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.linear = nn.Linear(hidden_dim, n_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x, l):
        x = self.embeddings(x)
        x = self.dropout(x)
        lstm_out, (ht, ct) = self.lstm(x)
        return self.linear(ht[-1])


class LSTM_fixed_word2vec(torch.nn.Module):
    def __init__(self, pre_trained_model, hidden_dim, n_classes):
        super().__init__()
        weights = torch.FloatTensor(pre_trained_model.wv.vectors)
        embedding = nn.Embedding.from_pretrained(weights)
        self.lstm = nn.LSTM(embedding.embedding_dim, hidden_dim, batch_first=True)
        self.linear = nn.Linear(hidden_dim, n_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.embedding(x)
        x = self.dropout(x)
        lstm_out, (ht, ct) = self.lstm(x)
        return self.linear(ht[-1])


class LSTM_variable_input(torch.nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, n_classes):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.dropout = nn.Dropout(0.3)
        self.embeddings = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.linear = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, s):
        x = self.embeddings(x)
        x = self.dropout(x)
        x_pack = pack_padded_sequence(x, s, batch_first=True, enforce_sorted=False)
        out_pack, (ht, ct) = self.lstm(x_pack)
        out = self.linear(ht[-1])
        return out


def train_model(model, train_dl, val_dl, epochs=10, lr=0.001):
    parameters = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=lr)
    min_val_loss = 1
    max_val_accuracy = -1
    for i in range(epochs):
        model.train()
        sum_loss = 0.0
        total = 0
        for inputs, target, l in train_dl:
            inputs = inputs.long()
            target = target.long()
            y_pred = model(inputs, l)
            optimizer.zero_grad()
            loss = F.cross_entropy(y_pred, target)
            loss.backward()
            optimizer.step()
            sum_loss += loss.item() * target.shape[0]
            total += target.shape[0]

        val_loss, val_acc, val_rmse = validation_metrics(model, val_dl)
        # plot_loss()
        print("epoch: %f/%f ->> train loss %f, val loss %f, val accuracy %f, and val rmse %.3f" % (
            i + 1, epochs, sum_loss / total, val_loss, val_acc, val_rmse))
        if (val_loss < min_val_loss) & (val_acc.item() > max_val_accuracy):
            min_val_loss = val_loss
            max_val_accuracy = val_acc.item()
            print('saving model -- best_checkpoint' + str(i + 1))
            torch.save({
                'epoch': i + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': sum_loss / total,
                'val_loss': val_loss
            }, 'best_checkpoint_10-2.pth')

        if i == (epochs - 1):
            print('saving last model -- last_checkpoint_LR' + str(i + 1))
            torch.save({
                'epoch': i + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': sum_loss / total,
                'val_loss': val_loss
            }, 'last_checkpoint_10-2' + str(i + 1) + '.pth')


def validation_metrics(model, valid_dl):
    model.eval()
    correct = 0
    total = 0
    sum_loss = 0.0
    sum_rmse = 0.0
    for x, y, l in valid_dl:
        x = x.long()
        y = y.long()
        y_hat = model(x, l)
        loss = F.cross_entropy(y_hat, y)
        pred = torch.max(y_hat, 1)[1]
        correct += (pred == y).float().sum()
        total += y.shape[0]
        sum_loss += loss.item() * y.shape[0]
        sum_rmse += np.sqrt(mean_squared_error(pred, y.unsqueeze(-1))) * y.shape[0]
    return sum_loss / total, correct / total, sum_rmse / total


In [13]:
data = pd.read_excel('temp_relevant_balanced.xlsx')
data = data[data.text.str.contains(' ', na=False)]
data['text_length'] = data['text'].apply(lambda x: len(x.split()))

# Zero-numbering the labels
data['DOCUMENT_TYPE'] = classToLabels(data['DOCUMENT_TYPE'])
# y = data['DOCUMENT_TYPE']
# X = data['text']
# counter = Counter(y)
# print(counter)
# over = SMOTE(sampling_strategy=0.1)
# under = RandomUnderSampler(sampling_strategy=0.5)
# X, y = over.fit_resample(X, y)
# X, y = under.fit_resample(X, y)
# counter = Counter(y)
# print(counter)
mean_words = int(np.ceil(np.mean(data['text_length'])))
print(mean_words)

# max_words = np.max(data['text_length'])
# print(max_words)

counts = Counter()
for index, row in data.iterrows():
    counts.update(row['text'].split())

# deleting infrequent words
# print("num_words before:", len(counts.keys()))
# for word in list(counts):
#     if counts[word] < 5:
#         del counts[word]
# print("num_words after:", len(counts.keys()))

# creating vocabulary
vocab2index = {"": 0, "UNK": 1}
words = ["", "UNK"]
for word in counts:
    vocab2index[word] = len(words)
    words.append(word)

data['encoded'] = data['text'].apply(lambda x: np.array(encode_sentence(x, vocab2index, mean_words)))
# data1 = data[data['DOCUMENT_TYPE']]

X = list(data['encoded'])
y = list(data['DOCUMENT_TYPE'])


186


In [19]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42, stratify=y)

# Separate test and validation.
# X_test, X_valid, y_test, y_valid = train_test_split(X_valid, y_valid, test_size=0.5, shuffle=True, random_state=42, stratify=y_valid)

train_ds = DocumentDataset(X_train, y_train)
valid_ds = DocumentDataset(X_valid, y_valid)
# test_ds = DocumentDataset(X_test, y_test)

batch_size = 500
vocab_size = len(words)
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_dl = DataLoader(valid_ds, batch_size=batch_size, shuffle=True)
# test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=True)

embedding_dim = 50
hidden_dim = 50
n_classes = 7

model_fixed = LSTM_fixed_len(vocab_size, embedding_dim=embedding_dim, hidden_dim=hidden_dim, n_classes=n_classes)
model_fixed.load_state_dict(torch.load('best_checkpoint_10-2.pth')['model_state_dict'])
print(model_fixed)
# train_model(model_fixed, train_dl, val_dl, epochs=50, lr=0.01)

LSTM_fixed_len(
  (embeddings): Embedding(112984, 50, padding_idx=0)
  (lstm): LSTM(50, 50, batch_first=True)
  (linear): Linear(in_features=50, out_features=7, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)


In [15]:
# print('Started Training...')
# train_model(model_fixed, train_dl, val_dl, epochs=10, lr=0.01)
# train_model(model_fixed, train_dl, val_dl, epochs=100, lr=0.001)

In [16]:
# train_model(model_fixed, train_dl, val_dl, epochs=100, lr=0.001)

In [ ]:
# model_fixed = LSTM_fixed_len(vocab_size, embedding_dim=embedding_dim, hidden_dim=hidden_dim, n_classes=n_classes)
# model_fixed.load_state_dict(torch.load('best_checkpoint.pth')['model_state_dict'])
# print(model_fixed)


In [20]:
# Metrics
y_true = []
y_pred = []
with torch.no_grad():
    for inputs, classes, i in val_dl:
        inputs = inputs.long()
        classes = classes.long()
        temp = model_fixed(inputs, 1)
        _, outputs = torch.max(temp, 1)
        for out in outputs:
            y_pred.append(out)
        for out in classes:
            y_true.append(out)

metrics = classification_report(y_true, y_pred)
print(metrics)

              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1048
           1       0.94      0.93      0.93      1300
           2       0.96      0.93      0.94      1240
           3       0.90      0.90      0.90      1300
           4       0.98      0.96      0.97      1100
           5       0.90      0.95      0.92       999
           6       0.99      0.99      0.99       999

    accuracy                           0.95      7986
   macro avg       0.95      0.95      0.95      7986
weighted avg       0.95      0.95      0.95      7986



In [21]:
final_obj = {'vocab2index': vocab2index, 'mean_words': mean_words,
             'vocab_size': vocab_size, 'embedding_dim': embedding_dim,
             'hidden_dim': hidden_dim, 'n_classes': n_classes}
json.dump(final_obj, open('vector_v1.txt', 'w'))